In [20]:
import os

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, col, explode, expr, lit, regexp_replace
from pyspark.sql.functions import round as spark_round
from pyspark.sql.functions import split
from pyspark.sql.functions import sum as spark_sum
from pyspark.sql.functions import udf, when
from pyspark.sql.types import (
    DateType,
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)
from pyspark.sql import Window
from pyspark.sql.functions import row_number

In [21]:
# Set the schema
listing_schema = StructType(
    [
        StructField("id", StringType(), True),
        StructField("name", StringType(), True),
        StructField("host_id", StringType(), True),
        StructField("host_name", StringType(), True),
        StructField("neighbourhood_group", StringType(), True),
        StructField("neighbourhood", StringType(), True),
        StructField("latitude", DoubleType(), True),
        StructField("longitude", DoubleType(), True),
        StructField("room_type", StringType(), True),
        StructField("price", DoubleType(), True),
        StructField("minimum_nights", IntegerType(), True),
        StructField("number_of_reviews", IntegerType(), True),
        StructField("last_review", DateType(), True),
        StructField("reviews_per_month", DoubleType(), True),
        StructField("calculated_host_listings_count", IntegerType(), True),
        StructField("availability_365", IntegerType(), True),
        StructField("number_of_reviews_ltm", IntegerType(), True),
        StructField("license", StringType(), True),
    ]
)

In [22]:
base_dir = "/mnt/data/public/insideairbnb/data.insideairbnb.com/united-states"
states = os.listdir(base_dir)
state_cities = {
    state: [city for city in os.listdir(os.path.join(base_dir, state))]
    for state in states
}

In [23]:
spark = SparkSession.builder.appName(f"Airbnb").getOrCreate()

city_dfs = {}

start_year = 2024
# for each state and list of cities
for state, cities in state_cities.items():
    # for each city in the list of cities
    for city in cities:
        print(state, city)
        # read all the files after year 2025
        df_data_list = []
        city_base_dir = os.path.join(base_dir, state, city)
        year_folders = [
            f
            for f in os.listdir(city_base_dir)
            if f[:4].isdigit() and int(f[:4]) >= start_year
        ]
        for year_folder in sorted(year_folders):
            data_filepath = os.path.join(
                city_base_dir, year_folder, "visualisations/listings.csv"
            )
            df_data = spark.read.csv(data_filepath, header=True, schema=listing_schema)
            df_data_list.append(df_data)

        # Combine the data
        df_data_combined = None
        df_data_combined = df_data_list[0]
        for df in df_data_list[1:]:
            df_data_combined = df_data_combined.unionByName(df)
        city_dfs[city] = df_data_combined

nc asheville
tx austin
tx dallas
tx fort-worth
ma boston
ma cambridge
il chicago
nv clark-county-nv
oh columbus
co denver
ca los-angeles
ca oakland
ca pacific-grove
ca san-diego
ca san-francisco
ca san-mateo-county
ca santa-clara-county
ca santa-cruz-county
fl broward-county
hi hawaii
nj jersey-city
nj newark
tn nashville
la new-orleans
ny new-york-city
ny albany
ny rochester
or portland
or salem-or
ri rhode-island
wa seattle
mn twin-cities-msa
dc washington-dc
mt bozeman


In [24]:
# union over all the cities with a new column just for which city
unioned_df = None

for city, df in city_dfs.items():
    # Add a new column with the city name
    df_with_city = df.withColumn("city", lit(city))

    if unioned_df is None:
        unioned_df = df_with_city
    else:
        # Union the new DataFrame
        unioned_df = unioned_df.unionByName(df_with_city)

# Show the first few rows
unioned_df.show(5, truncate=False)

+------+------------------------------------------+-------+---------+-------------------+-------------+--------+---------+---------------+-----+--------------+-----------------+-----------+-----------------+------------------------------+----------------+---------------------+-------+---------+
|id    |name                                      |host_id|host_name|neighbourhood_group|neighbourhood|latitude|longitude|room_type      |price|minimum_nights|number_of_reviews|last_review|reviews_per_month|calculated_host_listings_count|availability_365|number_of_reviews_ltm|license|city     |
+------+------------------------------------------+-------+---------+-------------------+-------------+--------+---------+---------------+-----+--------------+-----------------+-----------+-----------------+------------------------------+----------------+---------------------+-------+---------+
|155305|Cottage! BonPaul + Sharky's Hostel        |746673 |BonPaul  |NULL               |28806        |35.57864|

In [36]:
w = Window.partitionBy("id").orderBy("city")  # or any column for tie-breaking
core_attributes = unioned_df.withColumn("rn", row_number().over(w)) \
    .filter("rn = 1") \
    .drop("rn")

# compute avg price per id
avg_price_df = unioned_df.groupBy("id").agg(avg("price").alias("avg_price"))

# join so that each id appears once, keeping core attributes
listings_with_avg_price_unique = avg_price_df.join(core_attributes, on="id", how="left")

We will drop `neighbourhood_group` and `license` because over 45.0% and 36.0% of listings do not have any values for these.We also drop `last_review`, `reviews_per_month`, and `number_of_reviews` because these columns largely overlap with `number_of_reviews_ltm`.  By keeping only number_of_reviews_ltm, we capture both recent activity (last twelve months) and overall listing popularity, without retaining redundant review-related columns. Finally, we also dropped `id`, `name`, `host_id`, `host_name` since these are just identifiers for the listing. From here, we dropped all the rows that do not have the average price because this is the variable we will be predicting on. We are left with 333338 (84.0%) of the data.

In [52]:
filepath = "data/null_summary.csv"

# If the file does not exist yet, create the null summary
if not os.path.exists(filepath):
    total_rows = listings_with_avg_price_unique.count()
    null_counts_percentage = listings_with_avg_price_unique.select(
        [
            (
                spark_round(
                    1 - (spark_sum(when(col(c).isNull(), 1).otherwise(0)) / total_rows),
                    2,
                )
                * 100
            ).alias(f"{c}")
            for c in listings_with_avg_price_unique.columns
        ]
    )
    null_counts_percentage.toPandas().to_csv(filepath, index=False)

null_counts_df = pd.read_csv(filepath).T
null_counts_df.columns = ["Percentage Remaining (%)"]
null_counts_df

,Percentage Remaining (%)
id,100.0
avg_price,88.0
name,100.0
host_id,100.0
host_name,99.0
neighbourhood_group,45.0
neighbourhood,99.0
latitude,100.0
longitude,99.0
room_type,100.0


In [53]:
# Columns to drop
cols_to_drop = [
    "id",
    "host_id",
    "name",
    "host_name",
    "neighbourhood_group",
    "license",
    "last_review",
    "reviews_per_month",
    "number_of_reviews",
]

# Keep everything else
remaining_cols = [
    c for c in listings_with_avg_price_unique.columns if c not in cols_to_drop
]

# Create a cleaned DataFrame
dropped_listings_df = listings_with_avg_price_unique.select(*remaining_cols)


# save the dateframe if it doesnt exist yet
filepath = "data/cleaned_listings.parquet"
if not os.path.exists(filepath):
    dropped_listings_df.write.mode("overwrite").parquet(filepath)

# read the dataframe from a parquet
dropped_listings_df = spark.read.parquet(filepath)

# Show the aboslute and relative count of the remaining data points
total_count = dropped_listings_df.count()
remaining_count = dropped_listings_df.dropna().count()
print(f"Total Count is {total_count}.")
print(
    f"Remaining Count is {remaining_count} ({round(remaining_count/total_count * 100, 0)}%)."
)

Total Count is 396532.
Remaining Count is 333338 (84.0%).


In [55]:
cleaned_listings_df = dropped_listings_df.dropna()

In [78]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator

# 1️⃣ Drop 'neighbourhood'
df = cleaned_listings_df.drop("neighbourhood")

# 2️⃣ One-hot encode 'city'
indexer = StringIndexer(inputCol="city", outputCol="city_idx", handleInvalid="keep")
encoder = OneHotEncoder(inputCol="city_idx", outputCol="city_ohe")

# 3️⃣ Assemble features
feature_cols = [
    "latitude", "longitude", "room_type", "price",
    "minimum_nights", "calculated_host_listings_count",
    "availability_365", "number_of_reviews_ltm", "city_ohe"
]

# Room type is still string → index it
room_indexer = StringIndexer(inputCol="room_type", outputCol="room_type_idx", handleInvalid="keep")

assembler = VectorAssembler(
    inputCols=[c for c in feature_cols if c != "room_type"] + ["room_type_idx"],
    outputCol="features"
)

# 4️⃣ Linear Regression
lr = LinearRegression(featuresCol="features", labelCol="avg_price", maxIter=100, regParam=0.0, elasticNetParam=0.0)

# 5️⃣ Build pipeline
pipeline = Pipeline(stages=[room_indexer, indexer, encoder, assembler, lr])

# 6️⃣ Train/test split
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

# 7️⃣ Fit model
lr_model = pipeline.fit(train_df)

# 8️⃣ Make predictions
preds = lr_model.transform(test_df)
preds.select("avg_price", "prediction").show(5)

# 9️⃣ Evaluate
evaluator_rmse = RegressionEvaluator(labelCol="avg_price", predictionCol="prediction", metricName="rmse")
rmse = evaluator_rmse.evaluate(preds)

evaluator_r2 = RegressionEvaluator(labelCol="avg_price", predictionCol="prediction", metricName="r2")
r2 = evaluator_r2.evaluate(preds)

print(f"Linear Regression RMSE on test set: {rmse:.2f}")
print(f"Linear Regression R² on test set: {r2:.2f}")


+---------+------------------+
|avg_price|        prediction|
+---------+------------------+
|      8.0|  601.396998560027|
|      9.0|118.39449043573768|
|     10.0| 242.8473822240885|
|     13.0|16.922499509233603|
|     14.0|-1.645540575031191|
+---------+------------------+
only showing top 5 rows

Linear Regression RMSE on test set: 992.50
Linear Regression R² on test set: 0.83


In [79]:
pred_list = [row['prediction'] for row in preds.select('prediction').collect()]
pred_array = np.array(pred_list)

actual_list = [row['avg_price'] for row in test_df.select('avg_price').collect()]
actual_array = np.array(actual_list)

ForecastingMetrics.compute_all_metrics(actual_array, pred_array)

{'MAE': np.float32(158.55),
 'MSE': np.float32(985060.75),
 'RMSE': np.float32(992.50226),
 'MAPE': np.float32(95.62246),
 'SMAPE': np.float32(53.98845),
 'R2': np.float32(0.8262091),
 'Directional Accuracy': np.float32(52.419464)}

In [66]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator

# 1️⃣ Drop 'neighbourhood'
df = cleaned_listings_df.drop("neighbourhood")

# 2️⃣ One-hot encode 'city'
indexer = StringIndexer(inputCol="city", outputCol="city_idx", handleInvalid="keep")
encoder = OneHotEncoder(inputCol="city_idx", outputCol="city_ohe")

# 3️⃣ Assemble features
feature_cols = [
    "latitude", "longitude", "room_type", "price",
    "minimum_nights", "calculated_host_listings_count",
    "availability_365", "number_of_reviews_ltm", "city_ohe"
]

# Room type is still string → index it
room_indexer = StringIndexer(inputCol="room_type", outputCol="room_type_idx", handleInvalid="keep")

assembler = VectorAssembler(
    inputCols=[c for c in feature_cols if c != "room_type"] + ["room_type_idx"],
    outputCol="features"
)

# 4️⃣ Random Forest Regressor
rf = RandomForestRegressor(featuresCol="features", labelCol="avg_price", seed=42)

# 5️⃣ Build pipeline
pipeline = Pipeline(stages=[room_indexer, indexer, encoder, assembler, rf])

# 6️⃣ Train/test split
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

# 7️⃣ Fit model
rf_model = pipeline.fit(train_df)

# 8️⃣ Make predictions
preds = rf_model.transform(test_df)
preds.select("avg_price", "prediction").show(5)

# 9️⃣ Evaluate
evaluator = RegressionEvaluator(
    labelCol="avg_price", predictionCol="prediction", metricName="rmse"
)
rmse = evaluator.evaluate(preds)
print(f"RMSE on test set: {rmse:.2f}")


+---------+------------------+
|avg_price|        prediction|
+---------+------------------+
|      8.0|263.95358413020824|
|      9.0|473.37783736168274|
|     10.0| 299.1883952107976|
|     13.0| 248.8247896127858|
|     14.0|  198.230640655875|
+---------+------------------+
only showing top 5 rows

RMSE on test set: 1599.47


In [76]:
from typing import Any, Dict, Optional, Type, Union

import numpy as np
import pandas as pd


class ForecastingMetrics:
    """Class object that contains static methods for relevant
    forecasting metrics.

    Example Usage:
    --------------
    >>> from utils import ForecastingMetrics as forecast_metrics
    >>> forecast_metrics.mae(y_true, y_pred)
    """

    @staticmethod
    def mae(
        y_true: Union[np.ndarray, pd.Series], y_pred: Union[np.ndarray, pd.Series]
    ) -> np.float32:
        """Compute the mean absolute error (MAE) of a model's predictions
        against the actual (true) values.
        """
        return np.float32(np.mean(np.abs(y_true - y_pred)))

    @staticmethod
    def mse(
        y_true: Union[np.ndarray, pd.Series], y_pred: Union[np.ndarray, pd.Series]
    ) -> np.float32:
        """Compute the mean squared error (MSE) of a model's predictions
        against the actual (true) values.
        """
        return np.float32(np.mean((y_true - y_pred) ** 2))

    @staticmethod
    def rmse(
        y_true: Union[np.ndarray, pd.Series], y_pred: Union[np.ndarray, pd.Series]
    ) -> np.float32:
        """Compute the root mean squared error (RMSE) of a model's predictions
        against the actual (true) values.
        """
        return np.float32(np.sqrt(np.mean((y_true - y_pred) ** 2)))

    @staticmethod
    def mape(
        y_true: Union[np.ndarray, pd.Series], y_pred: Union[np.ndarray, pd.Series]
    ) -> np.float32:
        """Compute the mean absolute error (MAPE), in %, of a model's predictions
        against the actual (true) values.
        """
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)

        nonzero_filter = y_true != 0  # avoid division by 0
        y_true = y_true[nonzero_filter]
        y_pred = y_pred[nonzero_filter]
        return np.float32(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)

    @staticmethod
    def smape(
        y_true: Union[np.ndarray, pd.Series], y_pred: Union[np.ndarray, pd.Series]
    ) -> np.float32:
        """Compute the symmetric mean absolute error (SMAPE), in %, of a
        model's predictions against the actual (true) values.
        """
        numerator = np.abs(y_true - y_pred)
        denominator = np.abs(y_true) + np.abs(y_pred)
        nonzero_filter = denominator != 0

        return np.float32(
            np.mean(numerator[nonzero_filter] / (denominator[nonzero_filter] / 2)) * 100
        )

    @staticmethod
    def direction_accuracy(
        y_true: Union[np.ndarray, pd.Series], y_pred: Union[np.ndarray, pd.Series]
    ) -> np.float32:
        """Compute the direction accuracy, in %, of your predictions against the
        actual(true) values.
        """
        true_direction = np.diff(y_true) > 0
        pred_direction = np.diff(y_pred) > 0
        return np.float32(np.mean(true_direction == pred_direction) * 100)

    # =================DELETE THESE COMMENTS AFTER EDITING===================
    # FEEL FREE TO EDIT THE METHOD BELOW TO INCLUDE YOUR OTHER METRICS
    # YOU MAY ADD MORE METRICS FOR FORECASTING IF YOU DEEM NECESSARY
    @staticmethod
    def r2(
        y_true: Union[np.ndarray, pd.Series], y_pred: Union[np.ndarray, pd.Series]
    ) -> np.float32:
        """Compute the R-squared (coefficient of determination) of the predictions."""
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)
        ss_res = np.sum((y_true - y_pred) ** 2)
        ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
        return np.float32(1 - ss_res / ss_tot)

    
    @classmethod
    def compute_all_metrics(
        cls, y_true: Union[np.ndarray, pd.Series], y_pred: Union[np.ndarray, pd.Series]
    ) -> Dict[str, np.float32]:
        """Return a dictionary containing all of the metrics found within this
        class.
        """
        return {
            "MAE": cls.mae(y_true, y_pred),
            "MSE": cls.mse(y_true, y_pred),
            "RMSE": cls.rmse(y_true, y_pred),
            "MAPE": cls.mape(y_true, y_pred),
            "SMAPE": cls.smape(y_true, y_pred),
            "R2": cls.r2(y_true, y_pred),
            "Directional Accuracy": cls.direction_accuracy(y_true, y_pred),
        }


In [77]:
pred_list = [row['prediction'] for row in preds.select('prediction').collect()]
pred_array = np.array(pred_list)

actual_list = [row['avg_price'] for row in test_df.select('avg_price').collect()]
actual_array = np.array(actual_list)

ForecastingMetrics.compute_all_metrics(actual_array, pred_array)

{'MAE': np.float32(249.41498),
 'MSE': np.float32(2.5583092e+06),
 'RMSE': np.float32(1599.4716),
 'MAPE': np.float32(114.42636),
 'SMAPE': np.float32(56.39011),
 'R2': np.float32(0.54864633),
 'Directional Accuracy': np.float32(55.718937)}